<h2>Readability and Unfairness in Terms of Service Documents</h2>
CSE 595 Course Project

Kristine McLaughlin

<h4>1. Data Preprocessing</h4>

In [2]:
import os
import re
import pandas as pd

In [ ]:
# Extract annotated clauses from tagged documents to build dataset
folder_path = 'TaggedDocuments'
data = []

for file in os.listdir(folder_path):
    file_path = os.path.join(folder_path, file)
    with open(file_path, "r", encoding="utf-8") as f:
        document_text = f.read()

    # Handling nested tags with a stack
    tag_stack = []
    buffer_text = ""
    clause_start = 0 # index where the clause text starts

    # Get all tags in document (open and close)
    tag_pattern = re.compile(r"<(/?)([a-zA-Z]+)(\d+)>") # <potential backslash, then letters, then a number>
    for tag in tag_pattern.finditer(document_text):
        tag_type = tag.group(1)  # '' for open, '/' for close
        category = tag.group(2)
        num = int(tag.group(3))
        tag_start, tag_end = tag.span() # start is index of tag start <, end is index after >

        # Opening tag - add tag to stack
        if tag_type == "":
            tag_stack.append((category, num))
            # Update where the text (should) start
            clause_start = tag_end

        # Closing tag
        if tag_type == "/" and (category, num) in tag_stack:
            inner_text = document_text[clause_start:tag_start].strip().lower()

            if inner_text:  # only record if there’s text
                for category, num in tag_stack:
                    data.append({
                        "file_name": file,
                        "category": category,
                        "fairness_score": num,
                        "text": inner_text
                    })

            # Remove the matching tag from the stack
            if tag_stack and tag_stack[-1] == (category, num):
                tag_stack.pop()
            else:
                # Handle mismatched nesting by removing first occurrence
                if (category, num) in tag_stack:
                    tag_stack.remove((category, num))
        
        # Update where the text (should) start
        # clause_start = tag_end

# Turn array into df
data = pd.DataFrame(data, columns=["file_name", "category", "fairness_score", "text"])
print(f"Extracted {len(data)} annotated clauses from {len(os.listdir(folder_path))} files")
print(data.head())

Extracted 1360 annotated clauses from 50 files
     file_name category  fairness_score  \
0  Spotify.xml      use               2   
1  Spotify.xml       ch               2   
2  Spotify.xml      use               2   
3  Spotify.xml       ch               2   
4  Spotify.xml      ter               3   

                                                text  
0  by signing up or otherwise using the spotify s...  
1  occasionally we may, in our discretion, make c...  
2  in some cases, we will notify you in advance, ...  
3  spotify reserves the right, in its absolute di...  
4  spotify reserves the right, in its absolute di...  


In [16]:
# Manual check: Save to csv to investigate
data.to_csv("data.csv", index=False)